# LDLR promoter variants: coordinates (GRCh38)

Of four candidate LDLR promoter variants originally scoped (`c.-101T>C`, `c.-120C>T`,
`c.-121T>C`, `c.-215A>G`), functional evidence (Duskova et al., Eur J Hum Genet 2015,
PMC4277481) and clinical classification data point to **`c.-120C>T`** and **`c.-121T>C`**
as the two worth carrying forward into AlphaGenome prediction.

These HGVS coding-sequence variants (relative to NM_000527.5) fall upstream of the mRNA
reference sequence (which only extends to c.-86), so they can't be mapped with the
standard transcript-only tools (Ensembl VEP / Mutalyzer both fail with an out-of-boundary
error). Coordinates below were instead pulled from ClinVar, which reports them relative to
the RefSeqGene/chromosome sequence, and cross-checked against the local hg38 FASTA.

LDLR is on the **plus strand** in GRCh38 (chr19), so the cDNA base changes match the
genomic strand directly (no complementing needed). LDLR itself spans chr19:11,089,418-11,133,830
(~44.4 kb, Ensembl gene ENSG00000130164).

| Variant      | dbSNP        | ClinGen     | GRCh38 (chr19, 1-based) | GRCh37 (chr19, 1-based) | ref | alt | ClinVar classification |
|--------------|--------------|-------------|--------------------------|--------------------------|-----|-----|-------------------------|
| c.-121T>C    | rs777716188  | CA10576264  | 11,089,428               | 11,200,104               | T   | C   | Conflicting classifications |
| c.-120C>T    | —            | —           | 11,089,429               | 11,200,105               | C   | T   | Uncertain significance (expert panel) |

Sources: ClinVar VariationID 226302 (c.-121T>C), 226299 (c.-120C>T). Canonical SPDI
positions are 0-based, hence +1 for the 1-based coordinates above.

In [5]:
variants = {
    'c.-121T>C': {'chrom': 'chr19', 'pos_hg38': 11_089_428, 'ref': 'T', 'alt': 'C'},
    'c.-120C>T': {'chrom': 'chr19', 'pos_hg38': 11_089_429, 'ref': 'C', 'alt': 'T'},
}
variants

{'c.-121T>C': {'chrom': 'chr19', 'pos_hg38': 11089428, 'ref': 'T', 'alt': 'C'},
 'c.-120C>T': {'chrom': 'chr19', 'pos_hg38': 11089429, 'ref': 'C', 'alt': 'T'}}

## Confirm ref bases against the local hg38 FASTA

Directly read the reference base at each GRCh38 position (and a bit of flanking sequence)
from `reference_genome/hg38.fa` using its `.fai` index, and check it matches the expected
ref allele from ClinVar.

In [6]:
FASTA = '/scratch/st-cdeboer-1/sambina/reference_genome/hg38.fa'
FAI = '/scratch/st-cdeboer-1/sambina/reference_genome/hg38.fa.fai'


def load_fai(fai_path):
    fai = {}
    with open(fai_path) as f:
        for line in f:
            name, length, offset, linebases, linewidth = line.split()
            fai[name] = (int(length), int(offset), int(linebases), int(linewidth))
    return fai


def fetch_seq(fasta_path, fai, chrom, pos, span=1):
    """pos is 1-based, inclusive start; returns `span` bases from the fasta."""
    length, offset, linebases, linewidth = fai[chrom]
    idx0 = pos - 1
    line_num = idx0 // linebases
    line_off = idx0 % linebases
    file_offset = offset + line_num * linewidth + line_off
    with open(fasta_path, 'rb') as f:
        f.seek(file_offset)
        raw = f.read(span + span // linebases + 2)
    return raw.decode().replace('\n', '')[:span]


fai = load_fai(FAI)

for name, v in variants.items():
    chrom, pos, ref, alt = v['chrom'], v['pos_hg38'], v['ref'], v['alt']
    ref_base = fetch_seq(FASTA, fai, chrom, pos)
    context = fetch_seq(FASTA, fai, chrom, pos - 10, span=21)
    match = 'OK' if ref_base == ref else 'MISMATCH'
    print(f"{name}: {chrom}:{pos} (GRCh38)  ref={ref} alt={alt}  fasta_base={ref_base}  [{match}]")
    print(f"   context {pos-10}-{pos+10}: {context[:10]}[{context[10]}]{context[11:]}")

c.-121T>C: chr19:11089428 (GRCh38)  ref=T alt=C  fasta_base=T  [OK]
   context 11089418-11089438: GCTAGAAACC[T]CACATTGAAA
c.-120C>T: chr19:11089429 (GRCh38)  ref=C alt=T  fasta_base=C  [OK]
   context 11089419-11089439: CTAGAAACCT[C]ACATTGAAAT


## Build a single 131,072 bp AlphaGenome input window (shared by both variants)

The local AlphaGenome PyTorch model (`predict_alphagenome_ldlr.py`) has a **fixed** input
length of 131,072 bp (2^17) -- not the 1 Mb used earlier, which was too large for this
model. One window, split around an anchor point at **c.-118** (ATG-relative numbering,
same convention as the variant coordinates above; genomic position chr19:11,089,431),
keeping the same 40:60 upstream:downstream ratio as before, scaled down:

- **52,429 bp upstream** of the anchor (40% of 131,072)
- **78,643 bp downstream** of the anchor (60% of 131,072; still comfortably captures the
  full LDLR gene body, chr19:11,089,418-11,133,830, which is only ~44.4 kb)
- Total window size: 52,429 + 78,643 = **131,072 bp**

Both `c.-121T>C` (chr19:11,089,428) and `c.-120C>T` (chr19:11,089,429) sit only 2-3 bp
upstream of the anchor, so both fall inside the same window, close to its midpoint --
just on the upstream side of the split. The window bounds are identical for both variants;
only `var_pos` / `var_pos_in_window` / `ref` / `alt` differ per variant.

Two extra columns record where the **anchor (c.-118)** and the **translation start site**
(ATG, c.1; genomic chr19:11,089,549) fall within the window, both as 0-based offsets from
`window_start` (same convention as `var_pos_in_window`). Note: c. numbering has no c.0, so
the ATG sits exactly 118 bp downstream of the anchor by definition -- `atg_pos_in_window -
anchor_pos_in_window == 118` is a sanity check on this, not new information.

In [ ]:
import os
import pandas as pd

ANCHOR_C = -118                                    # ATG-relative anchor position
ANCHOR_POS_HG38 = ANCHOR_C + 11_089_549            # chr19:11,089,431
ATG_C = 1                                          # translation start site, A of ATG
ATG_POS_HG38 = ATG_C + 11_089_548                  # chr19:11,089,549 (no c.0, so +548 not +549)

MODEL_LEN = 131_072                                # fixed AlphaGenome PyTorch model input length (2^17)
UPSTREAM_SIZE = 52_429                             # 40% of MODEL_LEN
DOWNSTREAM_SIZE = 78_643                           # 60% of MODEL_LEN

WINDOW_CHROM = 'chr19'
WINDOW_START = ANCHOR_POS_HG38 - UPSTREAM_SIZE     # 1-based inclusive
WINDOW_END = ANCHOR_POS_HG38 + DOWNSTREAM_SIZE - 1

OUTPUT_DIR = '/scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome/LDLR'
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert WINDOW_END - WINDOW_START + 1 == UPSTREAM_SIZE + DOWNSTREAM_SIZE == MODEL_LEN

anchor_pos_in_window = ANCHOR_POS_HG38 - WINDOW_START   # 0-based offset from window_start
atg_pos_in_window = ATG_POS_HG38 - WINDOW_START         # 0-based offset from window_start
assert atg_pos_in_window - anchor_pos_in_window == 118

rows = []
for name, v in variants.items():
    rows.append({
        'variant': name,
        'chrom': WINDOW_CHROM,
        'window_start': WINDOW_START,
        'window_end': WINDOW_END,
        'var_pos': v['pos_hg38'],
        'var_pos_in_window': v['pos_hg38'] - WINDOW_START,
        'anchor_pos_in_window': anchor_pos_in_window,
        'atg_pos_in_window': atg_pos_in_window,
        'ref': v['ref'],
        'alt': v['alt'],
    })

windows_df = pd.DataFrame(rows)
windows_df

In [8]:
# Save one combined TSV (both variants share the same window) and one per-variant TSV
combined_path = os.path.join(OUTPUT_DIR, 'LDLR_c-121T_C_c-120C_T_window.tsv')
windows_df.to_csv(combined_path, sep='\t', index=False)
print(f"Saved combined windows → {combined_path}")

safe_names = {'c.-121T>C': 'c.-121T_C', 'c.-120C>T': 'c.-120C_T'}
for name in variants:
    out_path = os.path.join(OUTPUT_DIR, f'LDLR_{safe_names[name]}_window.tsv')
    windows_df[windows_df['variant'] == name].to_csv(out_path, sep='\t', index=False)
    print(f"Saved {name} window → {out_path}")

Saved combined windows → /scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome/LDLR/LDLR_c-121T_C_c-120C_T_window.tsv
Saved c.-121T>C window → /scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome/LDLR/LDLR_c.-121T_C_window.tsv
Saved c.-120C>T window → /scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome/LDLR/LDLR_c.-120C_T_window.tsv


## Sanity check: read back the saved TSV and print the first 10 bases from the ATG

Read the saved TSV from disk (not the in-memory `windows_df`), reconstruct the genomic
position of the translation start site from `window_start + atg_pos_in_window`, and fetch
the first 10 bases from there directly from the hg38 FASTA. Should start with `ATG`.

In [10]:
tsv_df = pd.read_csv(combined_path, sep='\t')

row = tsv_df.iloc[0]
atg_genomic_pos = row['window_start'] + row['anchor_pos_in_window']  # 1-based genomic position
first_10 = fetch_seq(FASTA, fai, row['chrom'], atg_genomic_pos, span=10)

print(f"ATG genomic position (from TSV): {row['chrom']}:{atg_genomic_pos}")
print(f"First 10 bases from translation start site: {first_10}")

ATG genomic position (from TSV): chr19:11089431
First 10 bases from translation start site: CATTGAAATG
